In [1]:
# Detect GPU and auto-downgrade to ESM-2 150M on T4 (< 15GB VRAM)
import torch
import sys
sys.path.insert(0, "/kaggle/input/datasets/guojanus/modules")
sys.path.insert(0, "/kaggle/input/datasets/guojanus/modules/plm_allergen_predictor")

gpu_name = ""
vram_gb = 0
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}, VRAM: {vram_gb:.1f} GB")
else:
    print("No GPU detected, using CPU")

from plm_allergen_predictor import ESM2ModelSize

if vram_gb < 15:
    MODEL_SIZE = ESM2ModelSize.LARGE  # 150M params
    print(f"⚠️  T4 GPU detected (VRAM < 15GB), downgrading to ESM-2 150M")
else:
    MODEL_SIZE = ESM2ModelSize.XLARGE  # 650M params
    print(f"✓ {gpu_name} GPU detected, using ESM-2 650M")

GPU: Tesla T4, VRAM: 15.6 GB
✓ Tesla T4 GPU detected, using ESM-2 650M


In [2]:
!wget https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz

import os


!rm -rf /kaggle/working/cd-hit-v4.8.1-2019-0228
!tar -xf cd-hit-v4.8.1-2019-0228.tar.gz
os.chdir("/kaggle/working/cd-hit-v4.8.1-2019-0228")
!make -j 4
os.chdir("/kaggle/working")
!/kaggle/working/cd-hit-v4.8.1-2019-0228/cd-hit -h

--2026-05-02 02:36:54--  https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/35050301/216f6a00-3b6b-11e9-9fec-85005717b86a?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-02T03%3A23%3A09Z&rscd=attachment%3B+filename%3Dcd-hit-v4.8.1-2019-0228.tar.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-02T02%3A22%3A26Z&ske=2026-05-02T03%3A23%3A09Z&sks=b&skv=2018-11-09&sig=UvZl%2FtE4KWWWjl4BBUAMbJon0i9f1CzS3S6RVMY3Ip4%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3NzY4OTcxNCwibmJmIjoxNzc3Njg5NDE0LCJwYXRoIjoicmVsZW

In [3]:
!pip install biopython
# subprocess.run(["pip", "install", "-q", "peft>=0.7", "biopython>=1.81", "scikit-learn>=1.3", "tqdm>=4.0"], check=True)
        

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 1.8 MB/s eta 0:00:00


In [4]:
# Load allergen data and prepare dataset
import os
from plm_allergen_predictor import DataPreprocessor, LoRAConfig

ALLERGEN_FASTA_DIR = "/kaggle/input/datasets/guojanus/allergen1"  # Kaggle dataset path
NEGATIVE_FASTA_PATH = "/kaggle/input/datasets/guojanus/allergen/uniprot_sprot.fasta"

os.environ["PATH"] = os.environ["PATH"] + ":/kaggle/working/cd-hit-v4.8.1-2019-0228/"
print(os.environ["PATH"])
preprocessor = DataPreprocessor(
    allergen_fasta_dir=ALLERGEN_FASTA_DIR,
    negative_fasta_path=NEGATIVE_FASTA_PATH,
    max_seq_len=1022,
    random_seed=42,
)

positive_samples = preprocessor.load_positive_samples()
#positive_samples = preprocessor.deduplicate(positive_samples)
negative_samples = preprocessor.load_negative_samples(n_samples=len(positive_samples) )  # 3× 正样本，自动过滤过敏原
all_samples = preprocessor.deduplicate(positive_samples + negative_samples,True)
print(f"Total samples after dedup: {len(all_samples)}")
print(f"Positive: {sum(1 for s in all_samples if s.label == 1)}, Negative: {sum(1 for s in all_samples if s.label == 0)}")

/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin:/kaggle/working/cd-hit-v4.8.1-2019-0228/
Total samples after dedup: 3176
Positive: 1113, Negative: 2063


In [5]:
# Build model and train with LoRA
import transformers
import sys
import torch
sys.path.insert(0, "/kaggle/input/datasets/guojanus/modules")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# 2. 强力清理缓存
torch.cuda.empty_cache()
from plm_allergen_predictor import (
    PLMEncoder, AllergenClassificationHead, FullModel,
    TrainingConfig, TrainingEngine, LoRAConfig
)

lora_config = LoRAConfig(rank=8, alpha=16, dropout=0.05)
device = "cuda" if torch.cuda.is_available() else "cpu"

encoder = PLMEncoder(
    model_size=MODEL_SIZE,
    lora_config=lora_config,
    device=device,
    use_gradient_checkpointing=True,
)

classifier = AllergenClassificationHead(
    hidden_dim=encoder.get_hidden_dim(),
    dropout_rate=0.3,
)

model = FullModel(encoder=encoder, classifier=classifier).to(device)
print(f"Trainable parameters: {encoder.get_trainable_params():,}")

# Prepare dataset split
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_SIZE.value)
dataset_split = preprocessor.split_dataset(all_samples, tokenizer=tokenizer)

# Training config
config = TrainingConfig(
    num_epochs=20,
    batch_size=1,
    grad_accum_steps=32,
    lora_lr=2e-4,
    head_lr=1e-3,
    use_amp=True,
    checkpoint_dir="model_checkpoint/",
    early_stopping_patience=5,
)

engine = TrainingEngine(model=model, config=config, dataset_split=dataset_split)
history = engine.train()
print(f"Training complete. Best val_auc: {max(m.val_auc for m in history):.4f}")

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable parameters: 2,027,520


CD-HIT 聚类失败（返回码 1）
相似度划分失败，回退到随机划分
Epoch 1/20:   1%|▏         | 31/2223 [00:11<11:42,  3.12it/s, loss=0.6327]/kaggle/input/datasets/guojanus/modules/plm_allergen_predictor/training_engine.py:303: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  self.scheduler.step()


Training complete. Best val_auc: 0.9774


In [6]:
# Load best checkpoint and run inference
from plm_allergen_predictor import InferencePipeline


pipeline = InferencePipeline(
    checkpoint_path="model_checkpoint/best_model.pt",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

# Example: predict on test sequences
test_sequences = [s.sequence for s in dataset_split.test.samples[:10]]
test_ids = [s.sequence_id for s in dataset_split.test.samples[:10]]

predictions = pipeline.predict(test_sequences, sequence_ids=test_ids)

for pred in predictions:
    print(f"{pred.sequence_id}: prob={pred.allergen_probability:.4f}, risk={pred.risk_level}, conf={pred.confidence:.4f}")

# Export report
pipeline.export_report(predictions, "allergen_predictions.csv", format="csv")
pipeline.export_report(predictions, "allergen_predictions.json", format="json")
print("Reports exported: allergen_predictions.csv, allergen_predictions.json")

sp|Q4JK71|CHL18_DERPT: prob=0.6226, risk=MEDIUM, conf=0.2451
sp|P86742|PRVB3_MACNO: prob=0.7131, risk=HIGH, conf=0.4261
A8F8Z5: prob=0.0604, risk=LOW, conf=0.8792
tr|Q45R38|Q45R38_WHEAT: prob=0.6585, risk=MEDIUM, conf=0.3170
Q1QMM7: prob=0.0538, risk=LOW, conf=0.8924
P65298: prob=0.3949, risk=MEDIUM, conf=0.2102
UPI002187CFA9: prob=0.1765, risk=LOW, conf=0.6470
sp|A2IA89|XCAP1_XENCH: prob=0.7460, risk=HIGH, conf=0.4919
Q4USR1: prob=0.2206, risk=LOW, conf=0.5587
sp|Q9XG85|PROF1_PARJU: prob=0.7629, risk=HIGH, conf=0.5257
Reports exported: allergen_predictions.csv, allergen_predictions.json
